# Microscopy Morphometry — SegFormer-B0 and U-Net–ResNet34

**Research code · Dr. Gülsüm Meço**

Code extracted from the research notebook `morfometri`, preserving the original experimental implementation. This workflow trains image-segmentation models and computes **prediction-derived morphometric measurements** from microscopy masks.

**Input:** polygon-annotated COCO microscopy data (`train/`, `valid/`, `test/`; onion, seed and stem tissue classes). The dataset is **not included** in this repository.

**Usage:** in Google Colab, run the dataset upload/extraction cell, then the main experiment cell. Review paths, GPU and package versions for your environment.

**Publishing note:** execution outputs, source images, credentials and trained weights are excluded. This release documents research code; end-to-end reproducibility has not been verified from the public repository alone. 


In [ ]:
# ============================================================
# STEP 1 — UPLOAD + EXTRACT + DATASET CONTROL
# ============================================================

from google.colab import files
import os, zipfile, glob, json, shutil

print("own_polygonal_39 ZIP dosyasını seç...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print("\nYüklendi:", zip_name)

# Temiz klasör
EXTRACT_DIR = "/content/own_polygonal_39"

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

os.makedirs(EXTRACT_DIR, exist_ok=True)

# ZIP aç
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(EXTRACT_DIR)

print("ZIP açıldı.")

# COCO annotation dosyalarını bul
ann_files = glob.glob(
    EXTRACT_DIR + "/**/_annotations.coco.json",
    recursive=True
)

print("\nBulunan annotation dosyaları:")
for x in ann_files:
    print(x)

# train/valid/test kökünü otomatik bul
DATASET_ROOT = None

for ann in ann_files:
    folder = os.path.dirname(ann)

    if os.path.basename(folder) == "train":
        possible_root = os.path.dirname(folder)

        required = [
            os.path.join(possible_root, "train", "_annotations.coco.json"),
            os.path.join(possible_root, "valid", "_annotations.coco.json"),
            os.path.join(possible_root, "test", "_annotations.coco.json"),
        ]

        if all(os.path.exists(p) for p in required):
            DATASET_ROOT = possible_root
            break

if DATASET_ROOT is None:
    raise RuntimeError("train/valid/test yapısı bulunamadı.")

print("\nDATASET ROOT:")
print(DATASET_ROOT)

# ------------------------------------------------------------
# Dataset kontrolü
# ------------------------------------------------------------

for split in ["train", "valid", "test"]:

    ann_path = os.path.join(
        DATASET_ROOT,
        split,
        "_annotations.coco.json"
    )

    with open(ann_path, "r") as f:
        coco = json.load(f)

    print("\n==========================")
    print(split.upper())
    print("==========================")

    print("Image count      :", len(coco["images"]))
    print("Annotation count :", len(coco["annotations"]))

    print("Categories:")
    for c in coco["categories"]:
        print(c["id"], "->", c["name"])

    seg_types = {}

    for ann in coco["annotations"]:
        seg = ann.get("segmentation")

        if isinstance(seg, list):
            typ = "polygon/list"
        elif isinstance(seg, dict):
            typ = "RLE/dict"
        else:
            typ = str(type(seg))

        seg_types[typ] = seg_types.get(typ, 0) + 1

    print("Segmentation types:", seg_types)

print("\n✅ DATASET CONTROL COMPLETE")

In [ ]:
# ============================================================
# FINAL POLYGONAL EXPERIMENT
# SegFormer-B0 + U-Net-ResNet34
# Segmentation + Prediction-Derived Morphometry
# ============================================================

!pip -q install transformers segmentation-models-pytorch timm pycocotools scikit-image

import os
import json
import random
import shutil
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from pycocotools import mask as mask_utils
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import segmentation_models_pytorch as smp

from skimage import measure
from scipy.stats import spearmanr

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

DATASET_ROOT = "/content/own_polygonal_39"
OUT_DIR = "/content/final_polygonal_results"

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)

os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
SIZE = 512
BATCH_SIZE = 2
MAX_EPOCHS = 50
PATIENCE = 10
LR = 6e-5
WEIGHT_DECAY = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

CLASS_NAMES = {
    0: "background",
    1: "onion",
    2: "seed",
    3: "stem"
}

NUM_CLASSES = 4

# ------------------------------------------------------------
# 2. COCO DATASET
# ------------------------------------------------------------

class PolygonCOCOBase:

    def __init__(self, split):

        self.split = split
        self.split_dir = os.path.join(DATASET_ROOT, split)

        ann_path = os.path.join(
            self.split_dir,
            "_annotations.coco.json"
        )

        with open(ann_path, "r") as f:
            self.coco = json.load(f)

        self.images = self.coco["images"]

        self.anns_by_image = {}

        for ann in self.coco["annotations"]:
            self.anns_by_image.setdefault(
                ann["image_id"], []
            ).append(ann)

        categories = {
            x["id"]: x["name"].lower()
            for x in self.coco["categories"]
        }

        self.category_map = {}

        for cat_id, name in categories.items():

            if "onion" in name:
                self.category_map[cat_id] = 1

            elif "seed" in name:
                self.category_map[cat_id] = 2

            elif "stem" in name:
                self.category_map[cat_id] = 3

            else:
                self.category_map[cat_id] = None

    def __len__(self):
        return len(self.images)

    def decode_annotation(self, ann, h, w):

        seg = ann.get("segmentation", None)

        if seg is None:
            return None

        # Polygon
        if isinstance(seg, list):

            if len(seg) == 0:
                return None

            try:
                rles = mask_utils.frPyObjects(seg, h, w)
                rle = mask_utils.merge(rles)
                mask = mask_utils.decode(rle)
            except:
                return None

        # RLE
        elif isinstance(seg, dict):

            rle = seg.copy()

            if isinstance(rle.get("counts"), str):
                rle["counts"] = rle["counts"].encode("utf-8")

            try:
                mask = mask_utils.decode(rle)
            except:
                return None

        else:
            return None

        if mask.ndim == 3:
            mask = np.any(mask, axis=2)

        return mask.astype(bool)

    def get_raw(self, idx):

        info = self.images[idx]

        image_path = os.path.join(
            self.split_dir,
            info["file_name"]
        )

        image = Image.open(image_path).convert("RGB")

        w, h = image.size

        semantic = np.zeros(
            (h, w),
            dtype=np.uint8
        )

        anns = self.anns_by_image.get(
            info["id"], []
        )

        for ann in anns:

            sem_class = self.category_map.get(
                ann["category_id"]
            )

            if sem_class is None:
                continue

            binary = self.decode_annotation(
                ann, h, w
            )

            if binary is None:
                continue

            semantic[binary] = sem_class

        return image, semantic, info["file_name"]


# ------------------------------------------------------------
# 3. SEGFORMER DATASET
# ------------------------------------------------------------

SEG_CKPT = "nvidia/segformer-b0-finetuned-ade-512-512"

processor = SegformerImageProcessor.from_pretrained(
    SEG_CKPT,
    do_reduce_labels=False,
    do_resize=False
)

class SegFormerDataset(Dataset):

    def __init__(self, split):
        self.base = PolygonCOCOBase(split)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):

        image, mask, fname = self.base.get_raw(idx)

        image = image.resize(
            (SIZE, SIZE),
            Image.BILINEAR
        )

        mask = Image.fromarray(mask).resize(
            (SIZE, SIZE),
            Image.NEAREST
        )

        mask = np.array(mask, dtype=np.uint8)

        encoded = processor(
            images=image,
            segmentation_maps=mask,
            return_tensors="pt"
        )

        return (
            encoded["pixel_values"].squeeze(0),
            encoded["labels"].squeeze(0).long(),
            fname
        )


# ------------------------------------------------------------
# 4. U-NET DATASET
# ------------------------------------------------------------

IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
)

class UNetDataset(Dataset):

    def __init__(self, split):
        self.base = PolygonCOCOBase(split)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):

        image, mask, fname = self.base.get_raw(idx)

        image = image.resize(
            (SIZE, SIZE),
            Image.BILINEAR
        )

        mask = Image.fromarray(mask).resize(
            (SIZE, SIZE),
            Image.NEAREST
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        image = (
            image - IMAGENET_MEAN
        ) / IMAGENET_STD

        image = np.transpose(
            image,
            (2, 0, 1)
        )

        mask = np.asarray(
            mask,
            dtype=np.int64
        )

        return (
            torch.tensor(image).float(),
            torch.tensor(mask).long(),
            fname
        )


# ------------------------------------------------------------
# 5. LOADERS
# ------------------------------------------------------------

seg_train = SegFormerDataset("train")
seg_valid = SegFormerDataset("valid")
seg_test  = SegFormerDataset("test")

unet_train = UNetDataset("train")
unet_valid = UNetDataset("valid")
unet_test  = UNetDataset("test")

def make_loader(ds, shuffle=False):

    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=True
    )

seg_train_loader = make_loader(seg_train, True)
seg_valid_loader = make_loader(seg_valid, False)
seg_test_loader  = make_loader(seg_test, False)

unet_train_loader = make_loader(unet_train, True)
unet_valid_loader = make_loader(unet_valid, False)
unet_test_loader  = make_loader(unet_test, False)

print("\nDataset sizes:")
print("Train:", len(seg_train))
print("Valid:", len(seg_valid))
print("Test :", len(seg_test))


# ------------------------------------------------------------
# 6. METRICS
# ------------------------------------------------------------

def confusion_update(conf, target, pred):

    target = target.reshape(-1)
    pred = pred.reshape(-1)

    valid = (
        (target >= 0) &
        (target < NUM_CLASSES)
    )

    inds = (
        NUM_CLASSES * target[valid] +
        pred[valid]
    )

    conf += np.bincount(
        inds,
        minlength=NUM_CLASSES**2
    ).reshape(NUM_CLASSES, NUM_CLASSES)

    return conf


def metrics_from_conf(conf):

    rows = []

    for c in range(NUM_CLASSES):

        tp = conf[c, c]
        fp = conf[:, c].sum() - tp
        fn = conf[c, :].sum() - tp

        precision = tp / (tp + fp + 1e-7)
        recall = tp / (tp + fn + 1e-7)

        dice = (
            2 * tp /
            (2 * tp + fp + fn + 1e-7)
        )

        iou = (
            tp /
            (tp + fp + fn + 1e-7)
        )

        rows.append({
            "class_id": c,
            "class": CLASS_NAMES[c],
            "precision": precision,
            "recall": recall,
            "dice": dice,
            "iou": iou
        })

    df = pd.DataFrame(rows)

    fg = df[df.class_id > 0]

    pixel_acc = (
        np.trace(conf) /
        (conf.sum() + 1e-7)
    )

    overall = {
        "pixel_accuracy": pixel_acc,
        "macro_precision_fg": fg.precision.mean(),
        "macro_recall_fg": fg.recall.mean(),
        "macro_dice_fg": fg.dice.mean(),
        "macro_iou_fg": fg.iou.mean()
    }

    return df, overall


def image_metrics(gt, pred):

    dice_list = []
    iou_list = []

    present = [
        c for c in range(1, NUM_CLASSES)
        if np.any(gt == c)
    ]

    for c in present:

        g = gt == c
        p = pred == c

        inter = np.logical_and(g, p).sum()

        dice = (
            2 * inter /
            (g.sum() + p.sum() + 1e-7)
        )

        union = np.logical_or(g, p).sum()

        iou = inter / (union + 1e-7)

        dice_list.append(dice)
        iou_list.append(iou)

    return (
        np.mean(dice_list),
        np.mean(iou_list)
    )


# ------------------------------------------------------------
# 7. VALIDATION FUNCTION
# ------------------------------------------------------------

@torch.no_grad()
def validation_dice(model, loader, model_type):

    model.eval()

    conf = np.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        dtype=np.int64
    )

    for x, y, _ in loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        if model_type == "segformer":

            out = model(pixel_values=x)

            logits = F.interpolate(
                out.logits,
                size=y.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        else:

            logits = model(x)

        pred = logits.argmax(1)

        confusion_update(
            conf,
            y.cpu().numpy(),
            pred.cpu().numpy()
        )

    _, overall = metrics_from_conf(conf)

    return overall["macro_dice_fg"]


# ------------------------------------------------------------
# 8. TRAIN SEGFORMER
# ------------------------------------------------------------

print("\n" + "="*70)
print("TRAINING SEGFORMER-B0")
print("="*70)

seed_everything(SEED)

seg_model = SegformerForSemanticSegmentation.from_pretrained(
    SEG_CKPT,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
    id2label=CLASS_NAMES,
    label2id={
        v: k for k, v in CLASS_NAMES.items()
    }
).to(DEVICE)

seg_optimizer = torch.optim.AdamW(
    seg_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

seg_best_path = os.path.join(
    OUT_DIR,
    "segformer_best.pt"
)

seg_history = []

best_dice = -1
wait = 0

scaler = torch.cuda.amp.GradScaler(
    enabled=(DEVICE == "cuda")
)

for epoch in range(1, MAX_EPOCHS + 1):

    seg_model.train()

    running_loss = 0

    for x, y, _ in seg_train_loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        seg_optimizer.zero_grad()

        with torch.cuda.amp.autocast(
            enabled=(DEVICE == "cuda")
        ):

            out = seg_model(
                pixel_values=x,
                labels=y
            )

            loss = out.loss

        scaler.scale(loss).backward()
        scaler.step(seg_optimizer)
        scaler.update()

        running_loss += loss.item()

    train_loss = (
        running_loss /
        len(seg_train_loader)
    )

    val_dice = validation_dice(
        seg_model,
        seg_valid_loader,
        "segformer"
    )

    seg_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_macro_dice_fg": val_dice
    })

    print(
        f"SegFormer E{epoch:02d} | "
        f"Loss {train_loss:.4f} | "
        f"Val Dice {val_dice:.4f}"
    )

    if val_dice > best_dice:

        best_dice = val_dice
        wait = 0

        torch.save(
            seg_model.state_dict(),
            seg_best_path
        )

    else:

        wait += 1

        if wait >= PATIENCE:
            print("SegFormer early stopping.")
            break

pd.DataFrame(seg_history).to_csv(
    os.path.join(
        OUT_DIR,
        "segformer_training_history.csv"
    ),
    index=False
)

seg_best_epoch = int(
    pd.DataFrame(seg_history)
    .loc[
        pd.DataFrame(seg_history)
        ["val_macro_dice_fg"].idxmax(),
        "epoch"
    ]
)

seg_model.load_state_dict(
    torch.load(
        seg_best_path,
        map_location=DEVICE
    )
)

print(
    "\nBEST SEGFORMER:",
    "Epoch", seg_best_epoch,
    "| Val Dice =", round(best_dice, 4)
)


# ------------------------------------------------------------
# 9. TRAIN U-NET
# ------------------------------------------------------------

print("\n" + "="*70)
print("TRAINING U-NET + RESNET34")
print("="*70)

seed_everything(SEED)

unet_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES
).to(DEVICE)

criterion = torch.nn.CrossEntropyLoss()

unet_optimizer = torch.optim.AdamW(
    unet_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

unet_best_path = os.path.join(
    OUT_DIR,
    "unet_resnet34_best.pt"
)

unet_history = []

best_dice_u = -1
wait = 0

scaler_u = torch.cuda.amp.GradScaler(
    enabled=(DEVICE == "cuda")
)

for epoch in range(1, MAX_EPOCHS + 1):

    unet_model.train()

    running_loss = 0

    for x, y, _ in unet_train_loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        unet_optimizer.zero_grad()

        with torch.cuda.amp.autocast(
            enabled=(DEVICE == "cuda")
        ):

            logits = unet_model(x)
            loss = criterion(logits, y)

        scaler_u.scale(loss).backward()
        scaler_u.step(unet_optimizer)
        scaler_u.update()

        running_loss += loss.item()

    train_loss = (
        running_loss /
        len(unet_train_loader)
    )

    val_dice = validation_dice(
        unet_model,
        unet_valid_loader,
        "unet"
    )

    unet_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_macro_dice_fg": val_dice
    })

    print(
        f"U-Net E{epoch:02d} | "
        f"Loss {train_loss:.4f} | "
        f"Val Dice {val_dice:.4f}"
    )

    if val_dice > best_dice_u:

        best_dice_u = val_dice
        wait = 0

        torch.save(
            unet_model.state_dict(),
            unet_best_path
        )

    else:

        wait += 1

        if wait >= PATIENCE:
            print("U-Net early stopping.")
            break

pd.DataFrame(unet_history).to_csv(
    os.path.join(
        OUT_DIR,
        "unet_training_history.csv"
    ),
    index=False
)

unet_best_epoch = int(
    pd.DataFrame(unet_history)
    .loc[
        pd.DataFrame(unet_history)
        ["val_macro_dice_fg"].idxmax(),
        "epoch"
    ]
)

unet_model.load_state_dict(
    torch.load(
        unet_best_path,
        map_location=DEVICE
    )
)

print(
    "\nBEST U-NET:",
    "Epoch", unet_best_epoch,
    "| Val Dice =", round(best_dice_u, 4)
)


# ------------------------------------------------------------
# 10. TEST EVALUATION
# ------------------------------------------------------------

@torch.no_grad()
def evaluate_test(
    model,
    loader,
    model_type,
    model_name
):

    model.eval()

    conf = np.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        dtype=np.int64
    )

    per_image = []

    predictions = {}

    for x, y, fnames in loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        if model_type == "segformer":

            out = model(pixel_values=x)

            logits = F.interpolate(
                out.logits,
                size=y.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        else:

            logits = model(x)

        pred = logits.argmax(1)

        gt_np = y.cpu().numpy()
        pred_np = pred.cpu().numpy()

        confusion_update(
            conf,
            gt_np,
            pred_np
        )

        for i, fname in enumerate(fnames):

            dice_i, iou_i = image_metrics(
                gt_np[i],
                pred_np[i]
            )

            per_image.append({
                "image": fname,
                "dice": dice_i,
                "iou": iou_i
            })

            predictions[fname] = (
                pred_np[i].astype(np.uint8)
            )

    class_df, overall = metrics_from_conf(conf)

    image_df = pd.DataFrame(per_image)

    overall[
        "image_mean_dice"
    ] = image_df.dice.mean()

    overall[
        "image_sd_dice"
    ] = image_df.dice.std(ddof=1)

    overall[
        "image_mean_iou"
    ] = image_df.iou.mean()

    overall[
        "image_sd_iou"
    ] = image_df.iou.std(ddof=1)

    class_df.to_csv(
        os.path.join(
            OUT_DIR,
            f"{model_name}_class_metrics.csv"
        ),
        index=False
    )

    image_df.to_csv(
        os.path.join(
            OUT_DIR,
            f"{model_name}_image_metrics.csv"
        ),
        index=False
    )

    with open(
        os.path.join(
            OUT_DIR,
            f"{model_name}_overall_metrics.json"
        ),
        "w"
    ) as f:
        json.dump(
            {
                k: float(v)
                for k, v in overall.items()
            },
            f,
            indent=2
        )

    return overall, predictions


seg_results, seg_predictions = evaluate_test(
    seg_model,
    seg_test_loader,
    "segformer",
    "segformer"
)

unet_results, unet_predictions = evaluate_test(
    unet_model,
    unet_test_loader,
    "unet",
    "unet"
)


# ------------------------------------------------------------
# 11. SAVE ORIGINAL-RESOLUTION PREDICTIONS
# ------------------------------------------------------------

raw_test = PolygonCOCOBase("test")

ref_dir = os.path.join(
    OUT_DIR,
    "reference_masks_original"
)

seg_pred_dir = os.path.join(
    OUT_DIR,
    "segformer_predictions_original"
)

unet_pred_dir = os.path.join(
    OUT_DIR,
    "unet_predictions_original"
)

os.makedirs(ref_dir, exist_ok=True)
os.makedirs(seg_pred_dir, exist_ok=True)
os.makedirs(unet_pred_dir, exist_ok=True)

original_data = {}

for i in range(len(raw_test)):

    image, ref_mask, fname = raw_test.get_raw(i)

    w, h = image.size

    safe_name = (
        os.path.splitext(
            os.path.basename(fname)
        )[0]
        + ".png"
    )

    Image.fromarray(
        ref_mask.astype(np.uint8)
    ).save(
        os.path.join(
            ref_dir,
            safe_name
        )
    )

    seg_512 = seg_predictions[fname]

    seg_original = np.array(
        Image.fromarray(seg_512).resize(
            (w, h),
            Image.NEAREST
        ),
        dtype=np.uint8
    )

    unet_512 = unet_predictions[fname]

    unet_original = np.array(
        Image.fromarray(unet_512).resize(
            (w, h),
            Image.NEAREST
        ),
        dtype=np.uint8
    )

    Image.fromarray(seg_original).save(
        os.path.join(
            seg_pred_dir,
            safe_name
        )
    )

    Image.fromarray(unet_original).save(
        os.path.join(
            unet_pred_dir,
            safe_name
        )
    )

    original_data[fname] = {
        "reference": ref_mask,
        "segformer": seg_original,
        "unet": unet_original
    }


# ------------------------------------------------------------
# 12. MORPHOMETRY
# ------------------------------------------------------------

def morphometry(binary_mask):

    binary_mask = binary_mask.astype(bool)

    area = float(binary_mask.sum())

    if area == 0:

        return {
            "area": 0.0,
            "area_fraction": 0.0,
            "component_count": 0,
            "total_perimeter": 0.0,
            "equivalent_diameter": 0.0,
            "major_axis": np.nan,
            "minor_axis": np.nan,
            "aspect_ratio": np.nan,
            "eccentricity": np.nan,
            "circularity": np.nan,
            "solidity": np.nan,
            "extent": np.nan,
            "largest_component_share": np.nan,
            "boundary_contact": 0
        }

    h, w = binary_mask.shape

    labelled = measure.label(
        binary_mask,
        connectivity=2
    )

    props = measure.regionprops(labelled)

    component_count = len(props)

    total_perimeter = float(
        sum(p.perimeter for p in props)
    )

    equivalent_diameter = math.sqrt(
        4 * area / math.pi
    )

    largest = max(
        props,
        key=lambda x: x.area
    )

    major = float(
        largest.axis_major_length
    )

    minor = float(
        largest.axis_minor_length
    )

    aspect = (
        major / minor
        if minor > 0
        else np.nan
    )

    lp = float(largest.perimeter)

    circularity = (
        4 * math.pi * largest.area /
        (lp**2)
        if lp > 0
        else np.nan
    )

    coords = largest.coords

    boundary_contact = int(
        np.any(coords[:, 0] == 0) or
        np.any(coords[:, 0] == h-1) or
        np.any(coords[:, 1] == 0) or
        np.any(coords[:, 1] == w-1)
    )

    return {
        "area": area,
        "area_fraction": area / (h*w),
        "component_count": component_count,
        "total_perimeter": total_perimeter,
        "equivalent_diameter": equivalent_diameter,
        "major_axis": major,
        "minor_axis": minor,
        "aspect_ratio": aspect,
        "eccentricity": float(largest.eccentricity),
        "circularity": circularity,
        "solidity": float(largest.solidity),
        "extent": float(largest.extent),
        "largest_component_share":
            float(largest.area / area),
        "boundary_contact": boundary_contact
    }


MORPH_METRICS = [
    "area",
    "area_fraction",
    "component_count",
    "total_perimeter",
    "equivalent_diameter",
    "major_axis",
    "minor_axis",
    "aspect_ratio",
    "eccentricity",
    "circularity",
    "solidity",
    "extent",
    "largest_component_share"
]


def compare_morphometry(model_key):

    rows = []

    for fname, item in original_data.items():

        ref = item["reference"]
        pred = item[model_key]

        # Only classes present in reference mask
        for class_id in [1, 2, 3]:

            if not np.any(ref == class_id):
                continue

            ref_m = morphometry(
                ref == class_id
            )

            pred_m = morphometry(
                pred == class_id
            )

            row = {
                "image": fname,
                "class_id": class_id,
                "class": CLASS_NAMES[class_id]
            }

            for metric in MORPH_METRICS:

                rv = ref_m[metric]
                pv = pred_m[metric]

                row[f"ref_{metric}"] = rv
                row[f"pred_{metric}"] = pv

                if (
                    pd.notna(rv) and
                    pd.notna(pv)
                ):

                    row[
                        f"error_{metric}"
                    ] = pv - rv

                    row[
                        f"abs_error_{metric}"
                    ] = abs(pv - rv)

                    if rv != 0:
                        row[
                            f"ape_{metric}_pct"
                        ] = (
                            abs(pv-rv) /
                            abs(rv) * 100
                        )
                    else:
                        row[
                            f"ape_{metric}_pct"
                        ] = np.nan

                else:

                    row[
                        f"error_{metric}"
                    ] = np.nan

                    row[
                        f"abs_error_{metric}"
                    ] = np.nan

                    row[
                        f"ape_{metric}_pct"
                    ] = np.nan

            rows.append(row)

    comparison = pd.DataFrame(rows)

    summary_rows = []

    for group_name, group_df in [
        ("ALL", comparison)
    ] + [
        (
            cls,
            comparison[
                comparison["class"] == cls
            ]
        )
        for cls in ["onion", "seed", "stem"]
    ]:

        if len(group_df) == 0:
            continue

        for metric in MORPH_METRICS:

            ref_col = f"ref_{metric}"
            pred_col = f"pred_{metric}"

            valid = group_df[
                [ref_col, pred_col]
            ].dropna()

            if len(valid) == 0:
                continue

            errors = (
                valid[pred_col] -
                valid[ref_col]
            )

            abs_errors = abs(errors)

            nonzero = (
                valid[ref_col] != 0
            )

            mape = np.nan

            if nonzero.sum() > 0:

                mape = np.mean(
                    abs_errors[nonzero] /
                    abs(valid.loc[
                        nonzero,
                        ref_col
                    ])
                ) * 100

            rho = np.nan
            pval = np.nan

            if (
                len(valid) >= 3 and
                valid[ref_col].nunique() > 1 and
                valid[pred_col].nunique() > 1
            ):

                rho, pval = spearmanr(
                    valid[ref_col],
                    valid[pred_col]
                )

            summary_rows.append({
                "group": group_name,
                "metric": metric,
                "n": len(valid),
                "reference_mean":
                    valid[ref_col].mean(),
                "prediction_mean":
                    valid[pred_col].mean(),
                "mean_signed_error":
                    errors.mean(),
                "MAE":
                    abs_errors.mean(),
                "MAPE_percent":
                    mape,
                "spearman_rho":
                    rho,
                "spearman_p":
                    pval
            })

    summary = pd.DataFrame(
        summary_rows
    )

    return comparison, summary


seg_morph, seg_morph_summary = (
    compare_morphometry("segformer")
)

unet_morph, unet_morph_summary = (
    compare_morphometry("unet")
)

seg_morph.to_csv(
    os.path.join(
        OUT_DIR,
        "segformer_prediction_morphometry.csv"
    ),
    index=False
)

seg_morph_summary.to_csv(
    os.path.join(
        OUT_DIR,
        "segformer_morphometry_summary.csv"
    ),
    index=False
)

unet_morph.to_csv(
    os.path.join(
        OUT_DIR,
        "unet_prediction_morphometry.csv"
    ),
    index=False
)

unet_morph_summary.to_csv(
    os.path.join(
        OUT_DIR,
        "unet_morphometry_summary.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 13. PARAMETER COUNTS
# ------------------------------------------------------------

seg_params = sum(
    p.numel()
    for p in seg_model.parameters()
    if p.requires_grad
)

unet_params = sum(
    p.numel()
    for p in unet_model.parameters()
    if p.requires_grad
)


# ------------------------------------------------------------
# 14. FINAL SUMMARY
# ------------------------------------------------------------

print("\n")
print("="*70)
print("FINAL RESULTS")
print("="*70)

print("\nSEGFORMER-B0")
print("Best epoch:", seg_best_epoch)
print("Best Val Dice:", round(best_dice, 4))
print(
    "Test Pixel Accuracy:",
    round(seg_results["pixel_accuracy"], 4)
)
print(
    "Test Precision FG:",
    round(seg_results["macro_precision_fg"], 4)
)
print(
    "Test Recall FG:",
    round(seg_results["macro_recall_fg"], 4)
)
print(
    "Test Dice FG:",
    round(seg_results["macro_dice_fg"], 4)
)
print(
    "Test IoU FG:",
    round(seg_results["macro_iou_fg"], 4)
)
print(
    "Image Dice Mean ± SD:",
    f'{seg_results["image_mean_dice"]:.4f} ± '
    f'{seg_results["image_sd_dice"]:.4f}'
)
print(
    "Parameters:",
    f"{seg_params:,}"
)

print("\nU-NET + RESNET34")
print("Best epoch:", unet_best_epoch)
print("Best Val Dice:", round(best_dice_u, 4))
print(
    "Test Pixel Accuracy:",
    round(unet_results["pixel_accuracy"], 4)
)
print(
    "Test Precision FG:",
    round(unet_results["macro_precision_fg"], 4)
)
print(
    "Test Recall FG:",
    round(unet_results["macro_recall_fg"], 4)
)
print(
    "Test Dice FG:",
    round(unet_results["macro_dice_fg"], 4)
)
print(
    "Test IoU FG:",
    round(unet_results["macro_iou_fg"], 4)
)
print(
    "Image Dice Mean ± SD:",
    f'{unet_results["image_mean_dice"]:.4f} ± '
    f'{unet_results["image_sd_dice"]:.4f}'
)
print(
    "Parameters:",
    f"{unet_params:,}"
)


# Key morphometric summary
print("\n")
print("="*70)
print("PREDICTION-DERIVED MORPHOMETRY")
print("="*70)

key_metrics = [
    "area",
    "area_fraction",
    "total_perimeter",
    "equivalent_diameter",
    "aspect_ratio",
    "eccentricity",
    "circularity"
]

print("\nSEGFORMER")
display(
    seg_morph_summary[
        (seg_morph_summary["group"] == "ALL") &
        (seg_morph_summary["metric"].isin(key_metrics))
    ][
        [
            "metric",
            "n",
            "MAE",
            "MAPE_percent",
            "spearman_rho"
        ]
    ]
)

print("\nU-NET")
display(
    unet_morph_summary[
        (unet_morph_summary["group"] == "ALL") &
        (unet_morph_summary["metric"].isin(key_metrics))
    ][
        [
            "metric",
            "n",
            "MAE",
            "MAPE_percent",
            "spearman_rho"
        ]
    ]
)


# ------------------------------------------------------------
# 15. SAVE EXPERIMENT INFO
# ------------------------------------------------------------

experiment_info = {
    "seed": SEED,
    "input_size": SIZE,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "segformer_best_epoch": seg_best_epoch,
    "unet_best_epoch": unet_best_epoch,
    "segformer_parameters": seg_params,
    "unet_parameters": unet_params
}

with open(
    os.path.join(
        OUT_DIR,
        "experiment_info.json"
    ),
    "w"
) as f:
    json.dump(
        experiment_info,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 16. ZIP EVERYTHING
# ------------------------------------------------------------

zip_path = shutil.make_archive(
    "/content/final_polygonal_results",
    "zip",
    OUT_DIR
)

print("\n✅ TAMAMLANDI")
print("Sonuç ZIP:", zip_path)

from google.colab import files
files.download(zip_path)